In [1]:
import pandas as pd
import numpy as np

In [31]:
np.random.seed(1)

In [32]:
df = pd.DataFrame({
    "Subject": np.repeat(["S1", "S2", "S3"], 5),
    "Trial": list(range(1,6))*3,
    "RT": np.random.normal(loc=500, scale=50, size=15) # reaction times in ms
})

df.head()

,Subject,Trial,RT
0,S1,1,581.217268
1,S1,2,469.412179
2,S1,3,473.591412
3,S1,4,446.351569
4,S1,5,543.270381


##### Direct Column Assignment (df['new_col'] = ...)
- If the column does not exist, it is created.  
- If it exists, it is overwritten.   

The right-hand side must be broadcastable to the DataFrame length (scalar or aligned Series).


In [24]:
# Create log-transformed RT

df["log_RT"] = np.log(df["RT"])

df.head()

,Subject,Trial,RT,log_RT
0,S1,1,445.005437,6.098086
1,S1,2,491.378590,6.197215
2,S1,3,456.107079,6.122728
3,S1,4,502.110687,6.218821
4,S1,5,529.140761,6.271254


Functional Column Creation (assign())
- It returns a new DataFrame with additional or modified columns.  
- It does not mutate the original unless explicitly reassigned.
- It supports lambda functions.

In [25]:
df2 = df.assign(new_col="expression")
print(df2.head())

# Lambda
df3 = df.assign(log_RT=lambda d: np.log(d["RT"]))
print(df3.head())

# assign() + lambda allows chaining:
df4 = (
    df
    .assign(log_RT=lambda d: np.log(d["RT"]))
    .assign(z_log_RT=lambda d: (d["log_RT"] - d["log_RT"].mean()) / d["log_RT"].std())
)
print(df4.head())

  Subject  Trial          RT    log_RT     new_col
0      S1      1  445.005437  6.098086  expression
1      S1      2  491.378590  6.197215  expression
2      S1      3  456.107079  6.122728  expression
3      S1      4  502.110687  6.218821  expression
4      S1      5  529.140761  6.271254  expression
  Subject  Trial          RT    log_RT
0      S1      1  445.005437  6.098086
1      S1      2  491.378590  6.197215
2      S1      3  456.107079  6.122728
3      S1      4  502.110687  6.218821
4      S1      5  529.140761  6.271254
  Subject  Trial          RT    log_RT  z_log_RT
0      S1      1  445.005437  6.098086 -1.397426
1      S1      2  491.378590  6.197215 -0.129937
2      S1      3  456.107079  6.122728 -1.082356
3      S1      4  502.110687  6.218821  0.146320
4      S1      5  529.140761  6.271254  0.816758


#### Inplace Operations and Why Avoiding Them Can Be Safer
Many pandas methods offer inplace=True. It mutates the object directly rather than returning a copy.  
> df.drop(columns=["Trial"], inplace=True)

However:
- In modern pandas, many inplace operations still create a copy internally.
- They break method chaining.
- They obscure data lineage.
- They increase risk of unintended state corruption in multi-step pipelines.
- They complicate debugging because previous states are lost.  

Immutability (or at least pseudo-immutability via reassignment) is safer.
> df = df.drop(columns=["Trial"])

Note: avoid inplace=True unless memory constraints are severe and you are certain of side effects.

In [36]:
# Let's normalize RT per subject using a lambda function.
df5 = df.assign(
    RT_z=lambda d: d.groupby("Subject")["RT"] # The lambda operates per group.
                  .transform(lambda x: (x - x.mean()) / x.std()) # transform() preserves original index length.
)
# Statistically appropriate if you assume subject-specific baselines and want to remove between-subject variance.

print(df5.head())

  Subject  Trial          RT      RT_z
0      S1      1  581.217268  1.379192
1      S1      2  469.412179 -0.586432
2      S1      3  473.591412 -0.512958
3      S1      4  446.351569 -0.991856
4      S1      5  543.270381  0.712055


Note:
- Small sample sizes per subject inflate variance estimates.
- If RT distributions are skewed, log-transform before normalization.

In [45]:
# Let's add a “Condition” column with random assignment.

# Note: always fix a seed or generate assignments externally and store them.

df6 = df.assign(
    Condition= lambda d: np.random.choice(["A","B"], size=len(d))
)

print(df6.head())

  Subject  Trial          RT Condition
0      S1      1  581.217268         B
1      S1      2  469.412179         A
2      S1      3  473.591412         A
3      S1      4  446.351569         B
4      S1      5  543.270381         A


#### Documenting derived columns is critical for reproducibility

Derived variables such as log-transformed reaction times or within-subject normalized scores
must be explicitly documented, including transformation formulas, parameter choices,
and grouping logic. Undocumented transformations compromise interpretability,
replicability, and downstream statistical validity.

##### Detecting Missing Values (isna() and notna())
- isna() returns a Boolean mask indicating missing values (NaN, None).
- notna() returns the complement.

In [47]:
# Create a dataframe with nan valuse

df_nan = df.copy()

# Make a mask
nan_mask = np.random.rand(len(df_nan)) < 0.2

print("The number of missing valuse: ", nan_mask.sum())

df_nan.loc[nan_mask, "RT"] = np.nan

print(df_nan.head())

The number of missing valuse:  4
  Subject  Trial          RT
0      S1      1  581.217268
1      S1      2  469.412179
2      S1      3         NaN
3      S1      4  446.351569
4      S1      5         NaN


In [52]:
print(df_nan["RT"].isna())
print("\n", df_nan["RT"].notna())

# Count missing values:
print("\n Number of NaN values: ", df_nan["RT"].isna().sum())

0     False
1     False
2      True
3     False
4      True
5     False
6     False
7      True
8     False
9     False
10    False
11    False
12    False
13     True
14    False
Name: RT, dtype: bool

 0      True
1      True
2     False
3      True
4     False
5      True
6      True
7     False
8      True
9      True
10     True
11     True
12     True
13    False
14     True
Name: RT, dtype: bool

 Number of NaN values:  4


##### Removing Missing Data: dropna()
- dropna() removes rows (or columns) containing missing values.
    - Pros:
        - Simple and transparent.
        - Preserves observed data integrity.
        - No artificial data introduced.
    - Cons:
        - Reduces sample size (lower statistical power).
        - Can bias estimates if missingness is not MCAR (Missing Completely At Random).
        - Alters group balance in repeated-measures designs.

In [54]:
df_clean = df_nan.dropna(subset=["RT"])

print("Number of NaN valuse: ", df_clean["RT"].isna().sum())

Number of NaN valuse:  0


##### Filling Missing Data: fillna()
- fillna() replaces missing values with specified values.
    - Pros:
        - Maintains sample size.
        - Simple to implement.
    - Cons:
        - Underestimates variance.
        - Biases standard errors.
        - Ignores hierarchical structure (e.g., subjects).
        - Can distort inferential statistics.

In [56]:
df_filled = df.fillna({"RT": df_nan["RT"].mean()})

print("Number of NaN valuse: ", df_filled["RT"].isna().sum())

Number of NaN valuse:  0


##### Imputation Strategies

Common strategies:
- Mean imputation:
    - Replace NaN with global mean.
    - Assumes homogeneity.
    - Shrinks variance.
- Median imputation:
    - More robust to skew.
    - Still reduces variance.
- Group-wise mean (e.g., per subject):
    - Respects hierarchical structure.
    - Still deterministic and variance-reducing.
- Custom imputation:
    - Regression-based imputation.
    - KNN.
    - Multiple imputation (statistically principled but outside basic pandas scope).
- Each strategy implicitly encodes assumptions about the missingness mechanism:
    - MCAR
    - MAR (Missing At Random)
    - MNAR (Missing Not At Random)

Pandas provides mechanical tools; statistical validity depends on your modeling assumptions.

In [58]:
# Let's introduce missing RT values artificially by randomly set 20% of RT values to NaN in another way.

missing_indices = rng.choice(df.index, size=int(len(df) * 0.2), replace=False)
df_missing = df.copy()
df_missing.loc[missing_indices, "RT"] = np.nan

print("Number of NaN valuse: ",df_missing["RT"].isna().sum())

Number of NaN valuse:  3


In [59]:
df_dropped = df_missing.dropna(subset=["RT"])

# Check size difference:
print(len(df_missing), len(df_dropped))

15 12


Note: sample size decreases. If missingness is systematic, estimates will be biased.

In [60]:
# Let's fill missing data using mean per subject.

df_imputed = df_missing.copy()

df_imputed["RT"] = (
    df_imputed.groupby("Subject")["RT"] # Isolates each participant.
    .transform(lambda x: x.fillna(x.mean())) # Preserves original row count.
)
# Missing RT values are replaced with that subject’s mean RT.

In [61]:
# Compare descriptive statistics before and after imputation
print(df.describe())
print(df_dropped.describe())

          Trial          RT
count  15.00000   15.000000
mean    3.00000  496.192854
std     1.46385   62.832848
min     1.00000  384.923065
25%     2.00000  465.675917
50%     3.00000  483.879140
75%     4.00000  549.979427
max     5.00000  587.240588
           Trial          RT
count  12.000000   12.000000
mean    3.250000  499.076276
std     1.484771   59.744898
min     1.000000  384.923065
25%     2.000000  467.544048
50%     3.500000  484.164382
75%     4.250000  546.624904
max     5.000000  587.240588


In [62]:
# More explicitly focusing on RT:

stats = pd.DataFrame({
    "Original_mean": [df["RT"].mean()],
    "Dropped_mean": [df_dropped["RT"].mean()],
    "Imputed_mean": [df_imputed["RT"].mean()],
    "Original_std": [df["RT"].std()],
    "Dropped_std": [df_dropped["RT"].std()],
    "Imputed_std": [df_imputed["RT"].std()]
})

print(stats)

   Original_mean  Dropped_mean  Imputed_mean  Original_std  Dropped_std  \
0     496.192854    499.076276    503.009596     62.832848    59.744898   

   Imputed_std  
0    53.580541  


Note: Pros and Cons Summary

- Drop missing:
    - Statistically defensible under MCAR.
    - Reduces power.
    - Potential bias if MAR/MNAR.
- Global mean imputation:
    - Preserves N.
    - Biased variance.
    - Inflates Type I error risk in some contexts.
- Group-wise mean:
    - Better for hierarchical designs.
    - Still underestimates uncertainty.
    - Does not propagate imputation uncertainty.
- Advanced methods (not shown here):
    - Multiple imputation preserves variance structure.
    - Requires modeling assumptions.
    - More computationally involved.